In [17]:
# Imports 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

#Dataset path 
import sys
sys.path.append("../")
from my_modules.data_collection_module import load_seoul_bike_data

In [2]:
#Load the dataset from my modules
from my_modules.data_collection_module import load_seoul_bike_data
df = load_seoul_bike_data("../data/SeoulBikeData.csv")
df.head()

,date,rented_bike_count,hour,temperature_c,humidity,wind_speed,visibility,dew_point_temperature_c,solar_radiation,rainfall,snowfall,season,holiday,functioning_day
0,2017-12-01,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
1,2017-12-01,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
2,2017-12-01,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes
3,2017-12-01,107,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
4,2017-12-01,78,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes


In [3]:
# Combine date and hour into a single datetime column
df["datetime"] = df["date"] + pd.to_timedelta(df["hour"], unit="h")

# Set datetime as index and sort chronologically 
df = df.set_index('datetime').sort_index()

In [7]:
def make_features(df_raw: pd.DataFrame) -> pd.DataFrame:
    """Create a supervised learning feature set from the raw Seoul Bike dataframe."""
    df_feat = pd.DataFrame()
    # Target series
    s = df_raw["rented_bike_count"].asfreq("h").ffill()
    df_feat["y"] = s

    # Calendar features (Seasonality)
    idx = s.index
    df_feat["hour"] = idx.hour
    df_feat["dayofweek"] = idx.dayofweek
    df_feat["month"] = idx.month

    # Lags (Autocorrelation & Seasonality)
    for lag in [1, 2, 24, 168]:
        df_feat[f"lag_{lag}"] = s.shift(lag)

    # Rolling means (Only use past information)
    df_feat["roll_mean_24"] = s.shift(1).rolling(24).mean()
    df_feat["roll_mean_168"] = s.shift(1).rolling(168).mean()

    # Covariates
    for col in [ "temperature_c", "humidity","wind_speed", "visibility","dew_point_temperature_c","solar_radiation","rainfall","snowfall","season","holiday",
        "functioning_day"]:
        if col in df_raw.columns:
            df_feat[col] = df_raw[col].asfreq("h").ffill()

    # Remove rows with missing values
    df_feat.dropna(inplace=True)

    return df_feat


feat = make_features(df)
feat.head()


,y,hour,dayofweek,month,lag_1,lag_2,lag_24,lag_168,roll_mean_24,roll_mean_168,...,humidity,wind_speed,visibility,dew_point_temperature_c,solar_radiation,rainfall,snowfall,season,holiday,functioning_day
datetime,,,,,,,,,,,,,,,,,,,,,
2017-12-08 00:00:00,233,0,4,12,259.0,389.0,198.0,254.0,356.208333,342.488095,...,49,2.6,1893,-12.6,0.0,0.0,0.0,Winter,No Holiday,Yes
2017-12-08 01:00:00,230,1,4,12,233.0,259.0,218.0,204.0,357.666667,342.363095,...,50,2.5,1902,-12.6,0.0,0.0,0.0,Winter,No Holiday,Yes
2017-12-08 02:00:00,131,2,4,12,230.0,233.0,149.0,173.0,358.166667,342.517857,...,53,1.0,1918,-12.4,0.0,0.0,0.0,Winter,No Holiday,Yes
2017-12-08 03:00:00,99,3,4,12,131.0,230.0,102.0,107.0,357.416667,342.267857,...,52,1.6,1978,-13.1,0.0,0.0,0.0,Winter,No Holiday,Yes
2017-12-08 04:00:00,71,4,4,12,99.0,131.0,62.0,78.0,357.291667,342.220238,...,53,2.4,1936,-13.4,0.0,0.0,0.0,Winter,No Holiday,Yes


In [14]:
# Separate predictors and target
X = feat.drop(columns=["y"])
y = feat["y"]

# Encode season
X = pd.get_dummies(
    X,
    columns=["season"],
    dtype=int
)

# Encode holiday and functioning_day (drop first category)
X = pd.get_dummies(
    X,
    columns=["holiday", "functioning_day"],
    drop_first=True,
    dtype=int
)

X.head()

,hour,dayofweek,month,lag_1,lag_2,lag_24,lag_168,roll_mean_24,roll_mean_168,temperature_c,...,dew_point_temperature_c,solar_radiation,rainfall,snowfall,season_Autumn,season_Spring,season_Summer,season_Winter,holiday_No Holiday,functioning_day_Yes
datetime,,,,,,,,,,,,,,,,,,,,,
2017-12-08 00:00:00,0,4,12,259.0,389.0,198.0,254.0,356.208333,342.488095,-3.5,...,-12.6,0.0,0.0,0.0,0,0,0,1,1,1
2017-12-08 01:00:00,1,4,12,233.0,259.0,218.0,204.0,357.666667,342.363095,-3.7,...,-12.6,0.0,0.0,0.0,0,0,0,1,1,1
2017-12-08 02:00:00,2,4,12,230.0,233.0,149.0,173.0,358.166667,342.517857,-4.3,...,-12.4,0.0,0.0,0.0,0,0,0,1,1,1
2017-12-08 03:00:00,3,4,12,131.0,230.0,102.0,107.0,357.416667,342.267857,-4.8,...,-13.1,0.0,0.0,0.0,0,0,0,1,1,1
2017-12-08 04:00:00,4,4,12,99.0,131.0,62.0,78.0,357.291667,342.220238,-5.3,...,-13.4,0.0,0.0,0.0,0,0,0,1,1,1


In [15]:
# Use the last 168 hours for testing
h = 24 * 7

X_train = X.iloc[:-h]
X_test = X.iloc[-h:]

y_train = y.iloc[:-h]
y_test = y.iloc[-h:]

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (8424, 23)
Testing: (168, 23)


In [18]:
#Linear Regression 
lr = LinearRegression()
lr.fit(X_train, y_train)
fc_lr = lr.predict(X_test)

# Random Forest
rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
fc_rf = rf.predict(X_test)

In [19]:
# Linear Regression evaluation
mae_lr = mean_absolute_error(y_test, fc_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, fc_lr))
r2_lr = r2_score(y_test, fc_lr)

# Random Forest evaluation
mae_rf = mean_absolute_error(y_test, fc_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, fc_rf))
r2_rf = r2_score(y_test, fc_rf)

In [20]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "MAE": [mae_lr, mae_rf],
    "RMSE": [rmse_lr, rmse_rf],
    "R2": [r2_lr, r2_rf]
})

results

,Model,MAE,RMSE,R2
0,Linear Regression,130.867401,198.417527,0.706928
1,Random Forest,53.520337,76.138206,0.956846
